<a href="https://colab.research.google.com/github/raghad-cs/Esnad/blob/raghad-precedents/official_precedents_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install -q pandas fastparquet pyarrow sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 36.3 MB/s eta 0:00:00


In [17]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

project_dir = Path("/content/drive/MyDrive/Esnad")
artifacts_dir = project_dir / "artifacts" / "moj"
notebooks_dir = project_dir / "notebooks"

artifacts_dir.mkdir(parents=True, exist_ok=True)
notebooks_dir.mkdir(parents=True, exist_ok=True)

print(" تم إنشاء وتجهيز المجلدات في Google Drive:")
print(" مجلد المخرجات:", artifacts_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 تم إنشاء وتجهيز المجلدات في Google Drive:
 مجلد المخرجات: /content/drive/MyDrive/Esnad/artifacts/moj


In [18]:
import pandas as pd
import faiss
import numpy as np
import re
import json
from sentence_transformers import SentenceTransformer


MODEL_NAME = "BAAI/bge-m3"
EMBEDDING_DIMENSION = 1024

def normalize_arabic(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[\u064B-\u0652]', '', text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


csv_path = '/content/drive/MyDrive/Esnad/saudi_legal_cases.mvp.csv'

try:
    df = pd.read_csv(csv_path, encoding='cp1256')
except Exception:
    try:
        df = pd.read_csv(csv_path, encoding='utf-8-sig')
    except Exception:
        df = pd.read_csv('saudi_legal_cases.mvp.csv', encoding='cp1256')

# دمج وتنظيف النصوص
df['clean_text'] = df.apply(
    lambda row: normalize_arabic(
        f"عنوان القضية: {row.get('title', '')}. "
        f"الوقائع: {row.get('facts', '')}. "
        f"طلبات المدعي: {row.get('plaintiff_claims', '')}. "
        f"رد المدعى عليه: {row.get('defendant_response', '')}. "
        f"السبب الشرعي: {row.get('legal_reasoning', '')}. "
        f"الملخص: {row.get('summary', '')}."
    ), axis=1
)

print(f" تمت قراءة ومعالجة {len(df)} قضية/سابقة تجارية بنجاح!")


print(f"⏳ جاري تحميل نموذج الـ Embeddings ({MODEL_NAME})...")
model = SentenceTransformer(MODEL_NAME)

print("⏳ جاري تحويل النصوص إلى Embeddings...")
embeddings = model.encode(
    df['clean_text'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
).astype('float32')

index = faiss.IndexFlatIP(EMBEDDING_DIMENSION)
index.add(embeddings)


index_path = artifacts_dir / "precedents.index"
metadata_path = artifacts_dir / "precedents_metadata.parquet"
config_path = artifacts_dir / "moj_search_config.json"

faiss.write_index(index, str(index_path))
df.to_parquet(metadata_path, index=False)

moj_config = {
    "dataset_source": "MOJ Commercial Precedents CSV",
    "number_of_cases": len(df),
    "model_name": MODEL_NAME,
    "embedding_dimension": EMBEDDING_DIMENSION,
    "normalize_embeddings": True,
    "faiss_index_type": "IndexFlatIP",
    "index_filename": "precedents.index",
    "metadata_filename": "precedents_metadata.parquet"
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(moj_config, f, ensure_ascii=False, indent=4)

print("\n تم بحمد الله حفظ المخرجات الثلاثة داخل مجلد artifacts/moj/ في Google Drive:")
print(f"1 {index_path}")
print(f"2 {metadata_path}")
print(f"3 {config_path}")

 تمت قراءة ومعالجة 20 قضية/سابقة تجارية بنجاح!
⏳ جاري تحميل نموذج الـ Embeddings (BAAI/bge-m3)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

⏳ جاري تحويل النصوص إلى Embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


 تم بحمد الله حفظ المخرجات الثلاثة داخل مجلد artifacts/moj/ في Google Drive:
1 /content/drive/MyDrive/Esnad/artifacts/moj/precedents.index
2 /content/drive/MyDrive/Esnad/artifacts/moj/precedents_metadata.parquet
3 /content/drive/MyDrive/Esnad/artifacts/moj/moj_search_config.json
